# Проект - нейросеть для предсказания калорийности блюд

Данный ноутбук - развитие экспериментов `project_exp_v1`:
- основное отличие - добавление справочника калорийности ингредиентов как внешние данные
        - справочник нужно найти, обработать, сопоставить с ингредиентами и внедрить как-то в модель

- это должно помочь модели предсказывать калорийность всего блюда
    - так как задача сужается до определения массы ингредиентов в блюде
    - а в этой части серия 'Эксп_в1' дошла Mass MAE ~ 50
    - что дает надежду на общий итог предсказания калорийности тоже в райноне 50 MAE 
        - то есть достижение цели проекта

## Блок инициализации

In [1]:
# импорты
import pathlib as path
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# local scripts
import scripts.utils as utils
import scripts.models as models
import scripts.dataset as dataset

model_config = models.MealModelConfig()

SEED = model_config.SEED

# задание случайностей - seed
utils.set_random(SEED)

DATA_PATH = path.Path('data')

## Справочник калорийности

In [114]:
import scipy.sparse as sparse

ingr_cal_mapping_df = pd.read_csv(DATA_PATH / 'nutrients' / 'processed' / 'ingr_nutr_mapping_result.csv')
ingr_cal_mapping = ingr_cal_mapping_df[['food_ingr', 'energy_kcal']]

print('Ingredients kal mapping:', ingr_cal_mapping.shape)
print(ingr_cal_mapping.head(10))

Ingredients kal mapping: (197, 2)
        food_ingr  energy_kcal
0         almonds          598
1           apple           61
2      artichokes          160
3  chayote squash           19
4        bok choy           21
5       asparagus           20
6         avocado          160
7           bacon          449
8    turkey bacon          368
9      sandwiches          231


# План экспериментов

- Эксперимент 2.1 
    - использовать последнюю модель из "Серии 1" (эксп.9/эксп.13 - диапазон MAE 64-70)
    - добавить признаки калорийности из справочника 3-им входом во fusion-слой
        - как реализовать механизм отбора конкретных слов/токенов, чтобы подать именно матрицу текущих ингредиентов, а не все подряд

- Эксперимент 2.2
    - вместо fusion слоя по новому использовать механизм кросс-внимания - между признаками-ингредиентами и признаками картинки

- Эксперимент 2.3
    - развитие идей из "Серии 1" - убрать direct target голову?


## Эксперимент 2.1 - добавление справочника калорийности
- использовать последнюю модель из "Серии 1" (эксп.9/эксп.13 - диапазон MAE 64-70)
- добавить признаки калорийности из справочника 3-им входом во fusion-слой
    - как реализовать механизм отбора конкретных слов/токенов, чтобы подать именно матрицу текущих ингредиентов, а не все подряд

In [26]:
import importlib 
import scripts.dataset as dataset
import scripts.utils as utils
import scripts.models as models

importlib.reload(dataset)
importlib.reload(utils)
importlib.reload(models)

<module 'scripts.models' from 'd:\\Обучение\\DeepLearning\\Яндекс-DL\\Projects\\4-sprint-food-calories\\scripts\\models.py'>

In [ ]:
import numpy as np
import pandas as pd
import pathlib as path
from functools import partial
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

# local scripts
import scripts.dataset as dataset
import scripts.utils as utils
import scripts.models as models

config = models.MealModelConfig()

utils.set_random(config.SEED)
log = partial(utils.log_fn, config=config)

tokenizer = AutoTokenizer.from_pretrained(config.TEXT_MODEL_NAME)

train_dataset = dataset.MealDataset(model_config=config, split='train')
ingr2kal_ref_matrix, ingr_encoder = train_dataset.ingr2kal_ref_matrix, train_dataset.ingr_encoder

train_dataloader = DataLoader(dataset=train_dataset, batch_size=config.N_BATCH, shuffle=True, collate_fn=partial(dataset.collate_fn, tokenizer=tokenizer))

val_dataset = dataset.MealDataset(model_config=config, split='test', ingr2kal_ref_matrix=ingr2kal_ref_matrix, ingr_encoder=ingr_encoder)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=config.N_BATCH, shuffle=False, collate_fn=partial(dataset.collate_fn, tokenizer=tokenizer))

model = models.MealModelNN(config=config) 

# разморозка параметров
utils.unfreeze_model_params(model, config)

train_params = np.sum([p.numel() for p in model.parameters() if p.requires_grad])
print(f'trainable parameters = {train_params:,d}')

# подключаем обучаемые (активные) параметры к оптимизатору

optimizer = torch.optim.AdamW([
    {'params': [p for p in model.text_model.parameters() if p.requires_grad], 'lr': config.LR_TEXT_MODEL, 'weight_decay': config.WEIGHT_DECAY},
    {'params': [p for p in model.img_model.parameters() if p.requires_grad], 'lr': config.LR_IMG_MODEL, 'weight_decay': config.WEIGHT_DECAY},

    {'params': model.text_proj.parameters(), 'lr': config.LR_PROJ, 'weight_decay': config.WEIGHT_DECAY},
    {'params': model.img_proj.parameters(), 'lr': config.LR_PROJ, 'weight_decay': config.WEIGHT_DECAY},
    {'params': model.ref_k100_proj.parameters(), 'lr': config.LR_PROJ, 'weight_decay': config.WEIGHT_DECAY},
    
    {'params': model.norm_text.parameters(), 'lr': config.LR_PROJ, 'weight_decay': config.WEIGHT_DECAY},
    {'params': model.norm_img.parameters(), 'lr': config.LR_PROJ, 'weight_decay': config.WEIGHT_DECAY},
    {'params': model.norm_ref_k100.parameters(), 'lr': config.LR_PROJ, 'weight_decay': config.WEIGHT_DECAY},

    {'params': model.fusion.parameters(), 'lr': config.LR_FUSION, 'weight_decay': config.WEIGHT_DECAY},
    {'params': model.regression_head_k_100.parameters(), 'lr': config.LR_REGRESSION_HEAD, 'weight_decay': config.WEIGHT_DECAY},
    {'params': model.regression_head_mass.parameters(), 'lr': config.LR_REGRESSION_HEAD, 'weight_decay': config.WEIGHT_DECAY},
    {'params': model.regression_head_kal.parameters(), 'lr': config.LR_REGRESSION_HEAD, 'weight_decay': config.WEIGHT_DECAY}
])

# проверка, что все активные параметры добавлены в оптимизатор
utils.optimizer_params_sanity_check(model, optimizer)

# - прогрев learning rate warmup
iters_per_epoch = len(train_dataloader)

# for real
#warmup_steps = int(1.7 * iters_per_epoch)   # ≈1.7 эпохи

# for debug
warmup_steps = 75

print(f'warmup_steps = {warmup_steps}')
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: min(1.0, step / warmup_steps)  # 150 шагов ≈ 1.7 эпохи при BATCH=32
)

print(f'batch size = {config.N_BATCH}')
print(f'n_iter = {config.N_ITER}')
print(f'train iterations/batch = {len(train_dataloader)}')
print(f'val iterations/batch = {len(val_dataloader)}')

train_loss_history = []
train_mae_history = []
train_mae_kal_100_history = []
train_mae_mass_history = []
train_mae_direct_history = []
train_time_history = []

val_loss_history = []
val_mae_history = []
val_mae_kal_100_history = []
val_mae_mass_history = []
val_mai_direct_history = []
val_time_history = []

best_metric = 1000
best_epoch = -1
for epoch in range(config.N_EPOCHS):
    train_huber_loss, train_metric_mae, train_kal_100_mae, train_mass_mae, train_direct_mae, train_time = utils.train(model=model, config=config, optimizer=optimizer, dataloader=train_dataloader, scheduler=scheduler)
    train_loss_history.append(train_huber_loss)
    train_mae_history.append(train_metric_mae)
    train_mae_kal_100_history.append(train_kal_100_mae)
    train_mae_mass_history.append(train_mass_mae)
    train_mae_direct_history.append(train_direct_mae)
    train_time_history.append(train_time)

    val_loss_huber, val_metric_mae, val_kal_100_mae, val_mass_mae, val_direct_mae, val_time = utils.validate(model=model, config=config, dataloader=val_dataloader)
    val_loss_history.append(val_loss_huber)
    val_mae_history.append(val_metric_mae)
    val_mae_kal_100_history.append(val_kal_100_mae)
    val_mae_mass_history.append(val_mass_mae)
    val_mai_direct_history.append(val_direct_mae)
    val_time_history.append(val_time)

    print(f'epoch - {epoch}: train_loss = {train_huber_loss:.2f}/{train_time:.2f} sec, train mae = {train_metric_mae:.2f}, direct mae = {train_direct_mae:.2f}, kal_100 mae = {train_kal_100_mae:.2f}, mass mae = {train_mass_mae:.2f}, \n \
            val_loss = {val_loss_huber:.2f}/{val_time:.2f} sec, val_mae = {val_metric_mae:.2f}, direct mae = {val_direct_mae:.2f}, kal_100_mae = {val_kal_100_mae:.2f}, mass_mae = {val_mass_mae:.2f}')
       
    if val_metric_mae < best_metric:
        best_metric = val_metric_mae
        best_epoch = epoch
        utils.save_checkpoint(model, config, optimizer, scheduler, epoch, best_epoch, best_metric)

    # если метрика не уменьшается в течение config.PATIENCE эпох подряд, то останавливаем обучение
    if (epoch - best_epoch) >= config.PATIENCE:
        print(f'early stopping - metric is not increasing for [{config.PATIENCE}] epochs')
        break

    if val_metric_mae < config.TARGET_METRIC:
        print(f'!! target metric reached = {config.TARGET_METRIC}, stop training')
        break

# save train logs
log_path  = path.Path(config.MODEL_LOG_SAVE_PATH)
log_path.mkdir(exist_ok=True)

pd.DataFrame(train_loss_history, columns=['loss_huber']).to_csv(log_path / 'train_loss_history.csv', index=False)
pd.DataFrame(train_mae_history, columns=['metric_mae']).to_csv(log_path / 'train_mae_history.csv', index=False)
pd.DataFrame(train_mae_kal_100_history, columns=['metric_mae']).to_csv(log_path / 'train_mae_kal_100_history.csv', index=False)
pd.DataFrame(train_mae_mass_history, columns=['metric_mae']).to_csv(log_path / 'train_mae_mass_history.csv', index=False)
pd.DataFrame(train_mae_direct_history, columns=['metric_mae']).to_csv(log_path / 'train_mae_direct_history.csv', index=False)
pd.DataFrame(train_time_history, columns=['time']).to_csv(log_path / 'train_time_history.csv', index=False)

pd.DataFrame(val_loss_history, columns=['loss_huber']).to_csv(log_path / 'val_loss_history.csv', index=False)
pd.DataFrame(val_mae_history, columns=['metric_mae']).to_csv(log_path / 'val_mae_history.csv', index=False)
pd.DataFrame(val_mae_kal_100_history, columns=['metric_mae']).to_csv(log_path / 'val_mae_kal_100_history.csv', index=False)
pd.DataFrame(val_mae_mass_history, columns=['metric_mae']).to_csv(log_path / 'val_mae_mass_history.csv', index=False)
pd.DataFrame(val_mai_direct_history, columns=['metric_mae']).to_csv(log_path / 'val_mae_direct_history.csv', index=False)
pd.DataFrame(val_time_history, columns=['time']).to_csv(log_path / 'val_time_history.csv', index=False)

Ref ingredients kal mapping: (197, 2)
trainable parameters = 9,587,843
warmup_steps = 2
batch size = 32
n_iter = 30
train iterations/batch = 86
val iterations/batch = 16


training:   2%|▏         | 2/86 [00:26<18:36, 13.29s/it]


KeyboardInterrupt: 

## График обучения